# 📘 Lesson 12: WaveNet Dilated Convolutions & Residual Blocks

**Step-by-Step Interactive Homework Notebook** with modular code execution and detailed explanations.


### 🔹 Step 1

**Purpose**: Import required libraries and frameworks (e.g. PyTorch, NumPy, Sklearn, Hugging Face).

- Sets up the execution environment, random seeds, and GPU/MPS device acceleration if available.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import random

from sklearn.decomposition import PCA

random.seed(42)
torch.manual_seed(42)


### 🔹 Load Dataset
Build Vocabulary

**Purpose**: Data Ingestion and Exploration.

- Loads raw datasets into memory, inspects shape, distributions, and initial sample structures.


In [ ]:
dataset_path="/Users/mac/Desktop/Machine Learning/DL/DB/Names/names.txt"
with open(dataset_path, "r") as f:
    words=f.read().splitlines()
    
print("Total Names:", len(words))
print("First 10 Names:", words[:10])


chars=sorted(list(set("".join(words))))

stoi={c:i+1 for i, c in enumerate(chars)}
stoi["."]=0

itos={i:c for c, i in stoi.items()}

vocab_size=len(stoi)

print(f"Vocabulary Size:{vocab_size}")
print(stoi)


### 🔹 Shuffle and Split Dataset

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
random.shuffle(words)

n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

train_words = words[:n1]
val_words = words[n1:n2]
test_words = words[n2:]

print(f"Training Names  : {len(train_words)}")
print(f"Validation Names: {len(val_words)}")
print(f"Test Names      : {len(test_words)}")


### 🔹 Custom Dataset
Initial context: [..]

**Purpose**: PyTorch Dataset & DataLoader Pipeline.

- Wraps tensors in iterable batches, handles multi-threaded worker loading, dynamic collation, and shuffling.


In [ ]:
block_size=3

class NamesDataset(Dataset):
    def __init__(self, words, stoi, block_size):
        self.X=[]
        self.Y=[]
        
        for word in words:
            context=[0]*block_size


### 🔹 add "." to indicate the end of the word
Slide the context window

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
for ch in word+'.':
                target=stoi[ch]
                
                self.X.append(context.copy())
                self.Y.append(target)
                
                context=context[1:]+[target]
                
        self.X=torch.tensor(self.X, dtype=torch.long)
        self.Y=torch.tensor(self.Y, dtype=torch.long)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, index):
        return self.X[index], self.Y[index]


### 🔹 Dataset Object

**Purpose**: PyTorch Dataset & DataLoader Pipeline.

- Wraps tensors in iterable batches, handles multi-threaded worker loading, dynamic collation, and shuffling.


In [ ]:
train_dataset = NamesDataset(
    train_words,
    stoi,
    block_size
)

val_dataset = NamesDataset(
    val_words,
    stoi,
    block_size
)

test_dataset = NamesDataset(
    test_words,
    stoi,
    block_size
)


### 🔹 Dataloader

**Purpose**: PyTorch Dataset & DataLoader Pipeline.

- Wraps tensors in iterable batches, handles multi-threaded worker loading, dynamic collation, and shuffling.


In [ ]:
batch_size = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)


### 🔹 Test

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
X_batch, Y_batch = next(iter(train_loader))

print("Input Shape :", X_batch.shape)
print("Target Shape:", Y_batch.shape)

print("\nFirst Input:")
print(X_batch[0])

print("\nFirst Target:")
print(Y_batch[0])


### 🔹 MLP Model

**Purpose**: Model Architecture Definition.

- Defines the network structure, layer projections, activations, and the forward propagation computation graph.


In [ ]:
class MLP(nn.Module):
    def __init__(self, 
                vocab_size, 
                block_size, 
                embedding_dim,
                hidden_size, 
                num_hidden_layer
        ):
        super().__init__()


### 🔹 Character Embedding

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
self.embedding=nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim
        )
        
        layers=[]
        
        input_size=block_size*embedding_dim


### 🔹 Hidden Layers

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
for _ in range(num_hidden_layer):
            layers.append(
                nn.Linear(input_size, hidden_size)
            )   
            
            layers.append(
                nn.ReLU()
            )    
            input_size=hidden_size


### 🔹 output layer
x-> (batch_size, block_size)

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
layers.append(
            nn.Linear(hidden_size, vocab_size)
        )
        self.network=nn.Sequential(*layers)
        
        
    def forward(self, x):
        x=self.embedding(x)
        
        # (batch_size, block_size, embedding_dim)
        x=x.view(x.size(0), -1)
        
        # (batch_size, block_size * embedding_dim)
        logits = self.network(x)
        
        return logits


### 🔹 Model

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
embedding_dim = 20
hidden_size = 200
num_hidden_layers = 2

model = MLP(
    vocab_size=vocab_size,
    block_size=block_size,
    embedding_dim=embedding_dim,
    hidden_size=hidden_size,
    num_hidden_layer=num_hidden_layers
)

print(model)


### 🔹 Check input and Output shapes

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
X_batch, Y_batch = next(iter(train_loader))

print("Input Shape :", X_batch.shape)

logits = model(X_batch)

print("Output Shape:", logits.shape)


### 🔹 Weight Initialization
Xavier Initialization

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
def initialize_xavier(model):
    for layer in model.modules():
        if isinstance(layer, nn.Linear):
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)


### 🔹 Step 16

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
def initialize_kaiming(model):
    for layer in model.modules():
        if isinstance(layer, nn.Linear):
            nn.init.kaiming_uniform_(
                layer.weight,
                nonlinearity="relu"
            )
            nn.init.zeros_(layer.bias)
   
   
         
initialize_xavier(model)
print(model.network[0].weight[:5])


### 🔹 Loss Function
Training Function

**Purpose**: Loss Function & Optimizer Initialization.

- Configures optimization objective and update rule (e.g., Adam, SGD with momentum, weight decay).


In [ ]:
criterion=nn.CrossEntropyLoss()

def train_one_epoch(model, dataloader, optimizer, criterion, device):
    
    model.train()
    running_loss=0.0
    
    for X_batch, Y_batch in dataloader:
        
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, Y_batch)
        
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(dataloader)


### 🔹 Validation Function

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
def evaluate(model, dataloader, criterion, device):

    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for X_batch, Y_batch in dataloader:

            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            logits = model(X_batch)

            loss = criterion(logits, Y_batch)

            running_loss += loss.item()

    return running_loss / len(dataloader)


### 🔹 Select Device

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
print(device)


### 🔹 Move model to device
Optimizer

**Purpose**: Loss Function & Optimizer Initialization.

- Configures optimization objective and update rule (e.g., Adam, SGD with momentum, weight decay).


In [ ]:
model.to(device)

learning_rate = 0.01

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)


### 🔹 Complete Training Loop

**Purpose**: Training & Optimization Loop.

- **Forward Pass**: Compute model predictions and loss.

- **Backward Pass**: `loss.backward()` calculates gradients via automatic differentiation.

- **Optimizer Step**: `optimizer.step()` updates trainable weights; `optimizer.zero_grad()` clears gradients.


In [ ]:
epochs=20
train_losses=[]
val_losses=[]

for epoch in range(epochs):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_loss = evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Validation Loss: {val_loss:.4f}"
    )


### 🔹 Test the Model
Hyperparameter Tunning

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
test_loss=evaluate(
    model, test_loader, criterion, device
)
print(f"Test Loss: {test_loss:.4f}")



configs = [

    {
        "block_size": 3,
        "embedding_dim": 10,
        "hidden_size": 100,
        "num_hidden_layers": 1,
        "learning_rate": 0.01,
        "initialization": "xavier"
    },

    {
        "block_size": 3,
        "embedding_dim": 20,
        "hidden_size": 200,
        "num_hidden_layers": 2,
        "learning_rate": 0.01,
        "initialization": "kaiming"
    },

    {
        "block_size": 5,
        "embedding_dim": 20,
        "hidden_size": 300,
        "num_hidden_layers": 2,
        "learning_rate": 0.005,
        "initialization": "kaiming"
    },

    {
        "block_size": 5,
        "embedding_dim": 30,
        "hidden_size": 300,
        "num_hidden_layers": 3,
        "learning_rate": 0.003,
        "initialization": "xavier"
    }

]


### 🔹 Variable for saving results
Train the model

**Purpose**: PyTorch Dataset & DataLoader Pipeline.

- Wraps tensors in iterable batches, handles multi-threaded worker loading, dynamic collation, and shuffling.


In [ ]:
results = []
best_model = None
best_config = None
best_validation_loss = float("inf")

for config in configs:
    train_dataset = NamesDataset(
        train_words,
        stoi,
        config["block_size"]
    )

    val_dataset = NamesDataset(
        val_words,
        stoi,
        config["block_size"]
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=256,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=256,
        shuffle=False
    )

    model = MLP(
        vocab_size,
        config["block_size"],
        config["embedding_dim"],
        config["hidden_size"],
        config["num_hidden_layers"]
    ).to(device)


    if config["initialization"] == "xavier":
        initialize_xavier(model)
    else:
        initialize_kaiming(model)



    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"]
    )

    epochs = 20
    for epoch in range(epochs):

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device
        )

        validation_loss = evaluate(
            model,
            val_loader,
            criterion,
            device
        )

    results.append({
        "config": config,
        "validation_loss": validation_loss

    })

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_model = model
        best_config = config


### 🔹 Result

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
print("\nResults")

for result in results:
    print(result["config"])
    print(
        "Validation Loss:",
        round(result["validation_loss"], 4)
    )

    print("-" * 50)


### 🔹 Print the Best Configuration
Evaluate the Best Model on the Test Set

**Purpose**: PyTorch Dataset & DataLoader Pipeline.

- Wraps tensors in iterable batches, handles multi-threaded worker loading, dynamic collation, and shuffling.


In [ ]:
print("\nBest Configuration")
print(best_config)
print(f"Best Validation Loss: {best_validation_loss:.4f}")


test_dataset = NamesDataset(
    test_words,
    stoi,
    best_config["block_size"]
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False
)


### 🔹 evaluate

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
test_loss = evaluate(
    best_model,
    test_loader,
    criterion,
    device
)

print(f"Test Loss: {test_loss:.4f}")


### 🔹 Generate New Names

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
best_model.eval()
number_of_names=20
for _ in range(number_of_names):
    context = [0] * best_config["block_size"]
    generated_name = []
    
    while True:
        x=torch.tensor([context], dtype=torch.long).to(device)
        
        with torch.no_grad():
            logits=best_model(x)
            
        probabilities=F.softmax(logits, dim=1)
        
        index=torch.multinomial(
            probabilities, 
            num_samples=1
        ).item()
        
        context=context[1:]+[index]
        
        if index==0:
            break
        
        generated_name.append(itos[index])
        
    print("".join(generated_name))


### 🔹 Visualize Character Embeddings
Apply PCA
Plot

**Purpose**: Dataset Preprocessing & Feature Scaling.

- Splits data into Training/Validation/Testing sets to evaluate generalization.

- Standardizes features ($\mu=0, \sigma=1$) to stabilize gradient descent and prevent vanishing/exploding updates.


In [ ]:
embeddings = best_model.embedding.weight.detach().cpu().numpy()

pca = PCA(n_components=2)

reduced_embeddings = pca.fit_transform(embeddings)

plt.figure(figsize=(8,8))

for i in range(vocab_size):

    plt.scatter(
        reduced_embeddings[i,0],
        reduced_embeddings[i,1]
    )

    plt.text(
        reduced_embeddings[i,0],
        reduced_embeddings[i,1],
        itos[i],
        fontsize=12
    )

plt.title("Character Embeddings (PCA)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.grid(True)

plt.show()


### 🔹 Plot Validation Results

**Purpose**: Evaluation, Metrics & Visualization.

- Evaluates model accuracy, F1-scores, loss convergence curves, and error distributions.


In [ ]:
labels = []

losses = []

for i, result in enumerate(results):

    labels.append(f"Model {i+1}")

    losses.append(result["validation_loss"])

plt.figure(figsize=(8,5))

plt.bar(labels, losses)

plt.ylabel("Validation Loss")

plt.title("Hyperparameter Comparison")

plt.show()


### 🔹 Print final summary

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
print("=" * 60)
print("BEST MODEL")

print(best_config)

print(f"Validation Loss : {best_validation_loss:.4f}")
print(f"Test Loss       : {test_loss:.4f}")
print("=" * 60)


## 🎯 Summary & Key Takeaways
1. **Modular Execution**: Each component runs independently and validates intermediate tensor shapes and states.
2. **Core Insights**: Inspect the printed metrics, loss outputs, and visual distributions above.
3. **Next Lesson**: Applies these foundations to more advanced deep learning and transformer architectures.
